# Applied Data Science Project 1: Motor Vehicle Crashes in NYC

## Maggie Boles, Adina Kaplan, Logan McCorkle, & Chana Ochs 

## 03/28/2026

### The purpose of this code is to clean and process datasets from the following sources to be used later in our model
    Source 1 (NYC Motor Vehicle Crashes): https://catalog.data.gov/dataset/motor-vehicle-collisions-crashes
    Source 2 (Traffic Signal and Stop Requests): https://data.cityofnewyork.us/Transportation/Traffic-Signal-and-All-Way-Stop-Study-Requests/w76s-c5u4/about_data 
    Source 3 (Traffic Signal Condition): https://data.cityofnewyork.us/Social-Services/Traffic-Signal-Condition_4/e5mv-wy8f/about_data
    Source 4 (NYS DOT Maintained Traffic Signals): https://data.gis.ny.gov/datasets/c61a41873d4f4b06bae79c23ad98b9c6_0/about

#### 

In [41]:
# Create a formula to convert the Latitude and Longitude to a standardized form
# From WGS84 to EPSG:2263 using pyproj
# Import libraries
import pandas as pd
from pyproj import Transformer

def add_stateplane_2263(df, lat_col='LATITUDE', lon_col='LONGITUDE', 
                       x_new='X_STATEPLANE_2263', y_new='Y_STATEPLANE_2263'):
    """
    Adds EPSG:2263 (NY State Plane Long Island, feet) columns to a DataFrame.
    """
    # Create transformer: WGS84 (EPSG:4326) → EPSG:2263
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:2263", always_xy=True)
    
    # Convert (handle NaNs safely)
    valid = df[lat_col].notna() & df[lon_col].notna()
    
    x_2263 = pd.Series(index=df.index, dtype='float64')
    y_2263 = pd.Series(index=df.index, dtype='float64')
    
    if valid.any():
        x_vals, y_vals = transformer.transform(
            df.loc[valid, lon_col].values,   # Note: lon first for always_xy=True
            df.loc[valid, lat_col].values
        )
        x_2263.loc[valid] = x_vals
        y_2263.loc[valid] = y_vals
    
    df[x_new] = x_2263
    df[y_new] = y_2263
    
    print(f"Added {x_new} and {y_new} (EPSG:2263) to dataframe.")
    return df

In [42]:
#NYC Motor Vehicle Crashes
# Import Libraries
import pandas as pd
from datetime import datetime

# Load CSV
df = pd.read_csv('Motor_Vehicle_Collisions.csv') 

# 1. Standardize CRASH DATE → datetime.date (or full datetime)
df['CRASH DATE'] = pd.to_datetime(df['CRASH DATE'], errors='coerce').dt.date

# 2. Standardize CRASH TIME → datetime.time (or full datetime)
df['CRASH TIME'] = pd.to_datetime(df['CRASH TIME'], format='%H:%M', errors='coerce').dt.time

# 3. Create a full datetime column (very useful)
df['CRASH_DATETIME'] = pd.to_datetime(
    df['CRASH DATE'].astype(str) + ' ' + df['CRASH TIME'].astype(str),
    errors='coerce'
)

# 4. Extract Day of the Week
df['DAY_OF_WEEK'] = df['CRASH_DATETIME'].dt.day_name()        # e.g., "Monday", "Tuesday"
# OR numerical version (0 = Monday, 6 = Sunday)
df['DAY_OF_WEEK_NUM'] = df['CRASH_DATETIME'].dt.weekday

# 5. Add abbreviated day name
df['DAY_OF_WEEK_ABBR'] = df['CRASH_DATETIME'].dt.day_name().str[:3]

# 6. Create conversion from LAT/LON to FT. utilizing EPSG:2263
df = add_stateplane_2263(df, lat_col='LATITUDE', lon_col='LONGITUDE')

# Print a break
print()

# Check the results
print(df[['CRASH DATE', 'CRASH TIME', 'CRASH_DATETIME', 'DAY_OF_WEEK']].head())

# Save the cleaned version
df.to_csv('nyc_crashes_cleaned.csv', index=False)
print("\n✅ File saved as: nyc_crashes_cleaned.csv")

Added X_STATEPLANE_2263 and Y_STATEPLANE_2263 (EPSG:2263) to dataframe.

   CRASH DATE CRASH TIME      CRASH_DATETIME DAY_OF_WEEK
0  2021-09-11   02:39:00 2021-09-11 02:39:00    Saturday
1  2022-03-26   11:45:00 2022-03-26 11:45:00    Saturday
2  2023-11-01   01:29:00 2023-11-01 01:29:00   Wednesday
3  2022-06-29   06:55:00 2022-06-29 06:55:00   Wednesday
4  2022-09-21   13:21:00 2022-09-21 13:21:00   Wednesday

✅ File saved as: nyc_crashes_cleaned.csv


In [43]:
#NYC Motor Vehicle Crashes (cont.)
# 7. Drop rows with missing Latitude or Longitude 
df = df.dropna(subset=['LATITUDE', 'LONGITUDE']).reset_index(drop=True)

# 8. Filter data from 2022 to 03/28/2026 
start_date = pd.to_datetime('2022-01-01').date()
end_date = pd.to_datetime('2026-03-28').date()

df = df[
    (df['CRASH DATE'] >= start_date) & 
    (df['CRASH DATE'] <= end_date)
].reset_index(drop=True)

# Add Year and Month for easier analysis
df['YEAR'] = df['CRASH_DATETIME'].dt.year
df['MONTH'] = df['CRASH_DATETIME'].dt.month_name()

# 9. Create conversion from LAT/LON to FT. utilizing EPSG:2263
df = add_stateplane_2263(df, lat_col='LATITUDE', lon_col='LONGITUDE')

# Print a break
print()

# Check the results
print(f"Original rows: {len(pd.read_csv('nyc_crashes_cleaned.csv')):,}")  # for reference
print(f"Rows after cleaning and filtering: {len(df):,}")
print(f"Date range: {df['CRASH DATE'].min()} to {df['CRASH DATE'].max()}")
print("\nFirst 5 rows of key columns:")
print(df[['CRASH DATE', 'CRASH TIME', 'DAY_OF_WEEK', 'LATITUDE', 'LONGITUDE', 'YEAR']].head())

# Save the cleaned and filtered dataset
df.to_csv('nyc_crashes_cleaned_2022_to_2026.csv', index=False)
print("\n✅ File saved as: nyc_crashes_cleaned_2022_to_2026.csv")

Added X_STATEPLANE_2263 and Y_STATEPLANE_2263 (EPSG:2263) to dataframe.

Original rows: 655,771
Rows after cleaning and filtering: 229,626
Date range: 2022-01-01 to 2025-06-05

First 5 rows of key columns:
   CRASH DATE CRASH TIME DAY_OF_WEEK   LATITUDE  LONGITUDE  YEAR
0  2023-11-01   01:29:00   Wednesday  40.621790 -73.970024  2023
1  2022-09-22   16:16:00    Thursday  40.698257 -73.826320  2022
2  2022-07-12   17:50:00     Tuesday  40.663303 -73.960490  2022
3  2022-04-24   16:45:00      Sunday  40.607685 -74.138920  2022
4  2022-04-24   04:49:00      Sunday  40.855972 -73.869896  2022

✅ File saved as: nyc_crashes_cleaned_2022_to_2026.csv


In [44]:
#Traffic Signal and Stop Requests
# Import Libraries
import pandas as pd

# Load the dataset
df_requests = pd.read_csv('Traffic_Signal_Requests.csv')

print(f"Original shape: {df_requests.shape}")

# 1. Standardize Date Columns
date_columns = ['DateCreated', 'DateRequested', 'StatusDate', 'DateAssigned', 
                'TargetDate', 'ReEvaluationDate', 'SignalInstallDate', 
                'AW_InstallDate', 'TentativeSignalInstallDate', 'ATR_ReceiveDate', 
                'ATR_RequestDate', 'WarrantsSatisfiedDate']

for col in date_columns:
    if col in df_requests.columns:
        df_requests[col] = pd.to_datetime(df_requests[col], errors='coerce')

# 2. Extract useful time features
df_requests['REQUEST_YEAR'] = df_requests['DateRequested'].dt.year
df_requests['REQUEST_MONTH'] = df_requests['DateRequested'].dt.month_name()
df_requests['DAY_OF_WEEK'] = df_requests['DateRequested'].dt.day_name()

# 3. Clean Text Columns
text_cols = ['Borough', 'StatusDescription', 'StudyStatus', 'RequestType', 
             'LocationType', 'MainStreet', 'CrossStreet1', 'CrossStreet2']

for col in text_cols:
    if col in df_requests.columns:
        df_requests[col] = df_requests[col].astype(str).str.strip().str.upper()

# 4. Create a clean combined location field (very useful)
df_requests['FULL_LOCATION'] = (
    df_requests['MainStreet'].fillna('') + " @ " + 
    df_requests['CrossStreet1'].fillna('') + 
    df_requests['CrossStreet2'].apply(lambda x: f" & {x}" if pd.notna(x) and x != '' else '')
).str.strip()

# 5. Handle Boolean / Yes-No columns consistently
yes_no_cols = ['VisionZero', 'Seasonal', 'HurricaneSandy', 'SchoolHold', 
               'SchoolCrossing', 'Removal', 'MidBlock']

for col in yes_no_cols:
    if col in df_requests.columns:
        df_requests[col] = df_requests[col].astype(str).str.upper().map({
            'YES': True, 'Y': True, 'TRUE': True,
            'NO': False, 'N': False, 'FALSE': False
        }).fillna(False)

# 6. Filter to more recent / relevant requests (optional but recommended)
# Keep only requests from 2015 onward (adjust as needed)
df_requests = df_requests[df_requests['DateRequested'].dt.year >= 2015].reset_index(drop=True)

# 7. Create conversion from LAT/LON to FT. utilizing EPSG:2263
#df_requests = add_stateplane_2263(df_requests, lat_col='LATITUDE', lon_col='LONGITUDE')
#commenting out; next cell addresses coordinate issue

# Print a break
print()

# Final summary
print(f"Cleaned shape: {df_requests.shape}")
print(f"Date range (DateRequested): {df_requests['DateRequested'].min()} to {df_requests['DateRequested'].max()}")
print(f"Boroughs: {df_requests['Borough'].unique()}")

print("\nSample of key columns:")
print(df_requests[['Id', 'DateRequested', 'Borough', 'RequestType', 'StatusDescription', 
                   'FULL_LOCATION', 'REQUEST_YEAR', 'DAY_OF_WEEK']].head())

# Save cleaned version
df_requests.to_csv('Traffic_Signal_Requests_cleaned.csv', index=False)
print("\n✅ File saved as: Traffic_Signal_Requests_cleaned.csv")

Original shape: (74715, 57)

Cleaned shape: (35533, 61)
Date range (DateRequested): 2015-01-02 00:00:00 to 2026-02-06 00:00:00
Boroughs: <StringArray>
[       'QUEENS',         'BRONX',      'BROOKLYN',     'MANHATTAN',
 'STATEN ISLAND',       'VARIOUS',  'ALL BOROUGHS',             'N',
            'BQ',            'MQ',            'QM',             nan]
Length: 12, dtype: str

Sample of key columns:
       Id DateRequested    Borough     RequestType  \
0  228428    2025-07-28     QUEENS  TRAFFIC SIGNAL   
1  219220    2025-04-29      BRONX  TRAFFIC SIGNAL   
2  133369    2016-08-03     QUEENS  TRAFFIC SIGNAL   
3  230070    2025-10-16   BROOKLYN  TRAFFIC SIGNAL   
4  228624    2025-08-29  MANHATTAN  TRAFFIC SIGNAL   

                                   StatusDescription  \
0  ENGINEERING STUDY COMPLETED     (SIGNALS ENGIN...   
1                               STUDY REQUEST DENIAL   
2  ENGINEERING STUDY COMPLETED     (SIGNALS ENGIN...   
3  ENGINEERING STUDY COMPLETED     (SIGNALS EN

In [ ]:
# Traffic Signal and Stop Requests
# This Dataset needs coordinates added to be parsed into our formatted dataset for the model
# Here is our API request to create the X_2263 & Y_2263
# Import libraries
import pandas as pd
import requests
import time
from tqdm import tqdm   

# Load your cleaned requests file
df_requests = pd.read_csv('traffic_signal_requests_cleaned.csv')

# Your Geoclient API key
API_KEY = "XXX"  

def geocode_intersection(main_st, cross_st, borough, api_key):
    """Geocode NYC intersection using Geoclient API and return lat/lon + 2263 coords"""
    if pd.isna(cross_st) or str(cross_st).strip() == "":
        return None, None, None, None
    
    url = "https://api.nyc.gov/geoclient/v2/intersection.json"
    params = {
        "crossStreetOne": str(main_st).strip(),
        "crossStreetTwo": str(cross_st).strip(),
        "borough": str(borough).strip().upper(),
        "subscription-key": api_key
    }
    
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            if "intersection" in data:
                inter = data["intersection"]
                lat = inter.get("latitude")
                lon = inter.get("longitude")
                x2263 = inter.get("xCoordinate")      # Already in EPSG:2263 (feet)
                y2263 = inter.get("yCoordinate")
                return lat, lon, x2263, y2263
    except:
        pass
    return None, None, None, None

# Apply geocoding 
tqdm.pandas()   # nice progress bar

results = df_requests.progress_apply(
    lambda row: geocode_intersection(
        row.get('MainStreet'), 
        row.get('CrossStreet1'), 
        row.get('Borough'), 
        API_KEY
    ), axis=1
)

df_requests[['LATITUDE', 'LONGITUDE', 'X_2263', 'Y_2263']] = pd.DataFrame(results.tolist(), index=df_requests.index)

# Optional: Drop rows that failed to geocode
df_requests = df_requests.dropna(subset=['LATITUDE', 'LONGITUDE']).reset_index(drop=True)

print(f"Successfully geocoded: {len(df_requests)} rows")
print(df_requests[['MainStreet', 'CrossStreet1', 'Borough', 'X_2263', 'Y_2263']].head())

# Save the result
df_requests.to_csv('traffic_signal_requests_cleaned_with_2263.csv', index=False)

  3%|▎         | 1121/35533 [04:08<2:02:02,  4.70it/s]

In [ ]:
#Traffic Signal Condition
# Import Libraries
import pandas as pd

# Load the file
df_condition = pd.read_csv('Traffic_Signal_Condition.csv', low_memory=False)

print(f"Original shape: {df_condition.shape}")

# 1. Standardize Column Names (make them clean and consistent)
df_condition = df_condition.rename(columns={
    'Unique Key': 'UNIQUE_KEY',
    'Created Date': 'CREATED_DATE',
    'Closed Date': 'CLOSED_DATE',
    'Problem (formerly Complaint Type)': 'PROBLEM',
    'Problem Detail (formerly Descriptor)': 'PROBLEM_DETAIL',
    'Location Type': 'LOCATION_TYPE',
    'Incident Zip': 'INCIDENT_ZIP',
    'Incident Address': 'INCIDENT_ADDRESS',
    'Street Name': 'STREET_NAME',
    'Cross Street 1': 'CROSS_STREET_1',
    'Cross Street 2': 'CROSS_STREET_2',
    'Intersection Street 1': 'INTERSECTION_STREET_1',
    'Intersection Street 2': 'INTERSECTION_STREET_2',
    'X Coordinate (State Plane)': 'X_STATEPLANE',
    'Y Coordinate (State Plane)': 'Y_STATEPLANE',
    'Latitude': 'LATITUDE',
    'Longitude': 'LONGITUDE',
    'Borough': 'BOROUGH',
    'City': 'CITY',
    'Status': 'STATUS',
    'Due Date': 'DUE_DATE',
    'Updated Date': 'UPDATED_DATE',
    'Resolution Action': 'RESOLUTION_ACTION',
    'Community Board': 'COMMUNITY_BOARD'
})

# 2. Convert Date Columns to proper datetime
date_cols = ['CREATED_DATE', 'CLOSED_DATE', 'DUE_DATE', 'UPDATED_DATE']

for col in date_cols:
    if col in df_condition.columns:
        df_condition[col] = pd.to_datetime(df_condition[col], errors='coerce')

# 3. Create useful derived columns
df_condition['CREATED_YEAR'] = df_condition['CREATED_DATE'].dt.year
df_condition['CREATED_MONTH'] = df_condition['CREATED_DATE'].dt.month_name()
df_condition['CREATED_DAY_OF_WEEK'] = df_condition['CREATED_DATE'].dt.day_name()

# Time to close (in days) - very useful for analysis
if 'CLOSED_DATE' in df_condition.columns and 'CREATED_DATE' in df_condition.columns:
    df_condition['DAYS_TO_CLOSE'] = (df_condition['CLOSED_DATE'] - df_condition['CREATED_DATE']).dt.days

# 4. Clean Coordinate Columns
df_condition['LATITUDE'] = pd.to_numeric(df_condition['LATITUDE'], errors='coerce')
df_condition['LONGITUDE'] = pd.to_numeric(df_condition['LONGITUDE'], errors='coerce')
df_condition['X_STATEPLANE'] = pd.to_numeric(df_condition['X_STATEPLANE'], errors='coerce')
df_condition['Y_STATEPLANE'] = pd.to_numeric(df_condition['Y_STATEPLANE'], errors='coerce')

# Drop rows with missing lat/long (critical for mapping)
df_condition = df_condition.dropna(subset=['LATITUDE', 'LONGITUDE']).reset_index(drop=True)

# 5. Clean Text Columns
text_cols = ['BOROUGH', 'CITY', 'PROBLEM', 'PROBLEM_DETAIL', 'STATUS', 
             'LOCATION_TYPE', 'STREET_NAME', 'INCIDENT_ADDRESS']

for col in text_cols:
    if col in df_condition.columns:
        df_condition[col] = df_condition[col].astype(str).str.strip().str.upper()

# Create a clean location string for easier searching
df_condition['FULL_LOCATION'] = (
    df_condition['STREET_NAME'].fillna('') + 
    df_condition['CROSS_STREET_1'].apply(lambda x: f" @ {x}" if pd.notna(x) and x != '' and x != 'nan' else '')
).str.strip()

# 6. Filter to recent years for file size
# Keep only records from 2020 onward (change as needed)
df_condition = df_condition[df_condition['CREATED_DATE'].dt.year >= 2020].reset_index(drop=True)

# 7. Create conversion from LAT/LON to FT. utilizing EPSG:2263
df_condition = add_stateplane_2263(df_condition, lat_col='LATITUDE', lon_col='LONGITUDE')

# Print a break
print()

# Final summary
print(f"Cleaned shape: {df_condition.shape}")
print(f"Date range: {df_condition['CREATED_DATE'].min()} to {df_condition['CREATED_DATE'].max()}")
print(f"Unique Boroughs: {df_condition['BOROUGH'].unique()}")

print("\nSample of cleaned data:")
print(df_condition[['UNIQUE_KEY', 'CREATED_DATE', 'CLOSED_DATE', 'DAYS_TO_CLOSE', 
                    'BOROUGH', 'PROBLEM', 'FULL_LOCATION', 'LATITUDE', 'LONGITUDE']].head())

# Save the cleaned version
df_condition.to_csv('Traffic_Signal_Condition_cleaned.csv', index=False)
print("\n✅ File saved as: Traffic_Signal_Condition_cleaned.csv")